In [2]:
!pip install -U transformers datasets evaluate seqeval -q


In [3]:
import numpy as np
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

In [4]:
dataset = load_dataset("tomaarsen/conll2003")

dataset


DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [5]:
label_names = dataset["train"].features["ner_tags"].feature.names

label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [6]:
id2label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-ORG",
    4: "I-ORG",
    5: "B-LOC",
    6: "I-LOC",
    7: "B-MISC",
    8: "I-MISC"
}

label2id = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-MISC": 7,
    "I-MISC": 8
}

id2label

{0: 'O',
 1: 'B-PER',
 2: 'I-PER',
 3: 'B-ORG',
 4: 'I-ORG',
 5: 'B-LOC',
 6: 'I-LOC',
 7: 'B-MISC',
 8: 'I-MISC'}

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [8]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            # Special tokens
            if word_idx is None:
                label_ids.append(-100)

            # First token of word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            # Remaining subword tokens
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [9]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

In [10]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [11]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [12]:
metric = evaluate.load("seqeval")

In [13]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [
            label_names[p]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            label_names[l]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [14]:
training_args = TrainingArguments(

    output_dir="./bert-ner-model",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=5e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    push_to_hub=False
)


In [16]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.069002,0.067076,0.917118,0.926252,0.921662,0.980985
2,0.034702,0.069431,0.944013,0.937915,0.940954,0.984724
3,0.019428,0.061513,0.939711,0.943926,0.941814,0.985239


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.06378005372590881, metrics={'train_runtime': 461.7256, 'train_samples_per_second': 91.23, 'train_steps_per_second': 5.705, 'total_flos': 1050534559887048.0, 'train_loss': 0.06378005372590881, 'epoch': 3.0})

In [17]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.019428,0.061513,3,0.939711,0.943926,0.941814,0.985239


{'eval_loss': 0.06151261180639267,
 'eval_precision': 0.9397106109324759,
 'eval_recall': 0.9439260721335008,
 'eval_f1': 0.9418136245636021,
 'eval_accuracy': 0.9852387119562018}

In [23]:
with open("evaluation_results.txt", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")

In [19]:
trainer.save_model("./final-ner-model")

tokenizer.save_pretrained("./final-ner-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final-ner-model/tokenizer_config.json', './final-ner-model/tokenizer.json')

In [20]:
ner_pipeline = pipeline(
    "token-classification",
    model="./final-ner-model",
    tokenizer="./final-ner-model",
    aggregation_strategy="first"
)

sentence = "Sundar Pichai is the CEO of Google in California."

predictions = ner_pipeline(sentence)


predictions

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'PER',
  'score': np.float32(0.99914706),
  'word': 'Sundar Pichai',
  'start': 0,
  'end': 13},
 {'entity_group': 'ORG',
  'score': np.float32(0.9981477),
  'word': 'Google',
  'start': 28,
  'end': 34},
 {'entity_group': 'LOC',
  'score': np.float32(0.9978599),
  'word': 'California',
  'start': 38,
  'end': 48}]

In [22]:
with open("prediction_results.txt", "w") as f:
    for p in predictions:
        f.write(str(p) + "\n")

In [24]:

!zip -r final-ner-model.zip final-ner-model

from google.colab import files

files.download("evaluation_results.txt")
files.download("prediction_results.txt")
files.download("final-ner-model.zip")

updating: final-ner-model/ (stored 0%)
updating: final-ner-model/tokenizer.json (deflated 70%)
updating: final-ner-model/training_args.bin (deflated 53%)
updating: final-ner-model/tokenizer_config.json (deflated 44%)
updating: final-ner-model/config.json (deflated 55%)
updating: final-ner-model/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>